# CS336 07: one parent, compatible interventions

Follow a randomly initialized TinyDecoder through two equal-exposure SFT arms, then branch from the coverage-SFT checkpoint into accepted-response SFT and two sampled reward updates. The experiment asks whether delivered answers improve because the model learns the task, learns to stop, or receives broader source coverage.

This is an original symbolic parity assignment. The visible English prompts document the task; a ten-token symbolic interface encodes it. It does not parse natural language, inherit a Qwen checkpoint, or continue the historical constructive control. All3 predeclared seeds use24 SFT updates and16 fresh-rollout sibling updates, under a180-second total CPU cap. No held-out recipe tuning occurs.

Reuse: Chapters8–9's masks and SFT, Chapters11–12's frozen policies and sampling likelihoods, Chapters13–14's advantages and grading. The separate [reward audit](04_reward_signal_audit.ipynb) replays immutable native text without changing its original rewards.


## Checkpoint 1: interface and matched exposure

### Prediction

Predict the labels and logits positions for [BOS, operation, a, b, answer, EOS]. Which mask change would silently teach the prompt? At equal target exposure, how many unique sources do broad coverage and repetition actually contain? Compare both answer classes and paired held-out templates before fitting anything.


### Reference solution 1

Run the bounded reference below, then compare it with your prediction.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/dongxi_llms').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
from dongxi_llms import posttraining_assignment as pa
from dongxi_llms import reasoning_controls as rc
from dongxi_llms.grpo_lab import group_advantages, clipped_objective, exact_kl
from dongxi_llms.sft_lab import token_loss_sum
torch.set_num_threads(1)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True})
items, protocol = pa.load_fixture()
train = [i for i in items if i['split']=='train']
repeated = [i for i in train if (i['a'],i['b']) in ((0,0),(0,1))]*4
batch = pa.supervised_batch(train)
print(protocol['interface'])
print('Fixture audit:',pa.validate_fixture(items)); print('Baselines:',pa.baselines(items))
print('Unique sources:',len({i['source_group'] for i in train}),len({i['source_group'] for i in repeated}))
print('Per-update labels:',int((batch['labels']!=-100).sum()),int((pa.supervised_batch(repeated)['labels']!=-100).sum()))
wrong_labels = batch['labels'].clone()
wrong_labels[:,1:4] = batch['input_ids'][:,1:4]
print('Broken prompt-supervision target count:',int((wrong_labels[:,1:]!=-100).sum()))
fig,ax=plt.subplots(figsize=(10,3))
image=ax.imshow(batch['labels'][:4],vmin=-100,vmax=9,cmap='coolwarm',aspect='auto')
for i,row in enumerate(batch['labels'][:4].tolist()):
    for j,value in enumerate(row): ax.text(j,i,'masked' if value==-100 else str(value),ha='center',va='center')
ax.set_xticks(range(6),['BOS','operation','a','b','answer','EOS']); ax.set_yticks(range(4),[i['id'] for i in train[:4]])
ax.set(xlabel='Input/label position before causal shift',ylabel='Training example',title='Only answer and EOS targets are supervised in SFT')
ax.grid(False);plt.tight_layout();plt.show()


### Reference explanation 1

The reusable loss shifts once: logits at position3 predict the answer label at4; logits at4 predict EOS at5. Prompt labels remain−100. The deliberately broken mask adds operation and operand targets, changing the objective and budget from32 to80 labels per update. It is diagnosed rather than used for training.

Coverage has8 underlying sources×2 templates; repetition has2 sources×2 templates×4 repeats. Both arms see16 examples and32 targets per update, or768 targets across24 updates. Both classes have multiple source groups in every split. Direct and alias forms share the same held-out sources; both templates appear during training, so their contrast isolates serialization effects on these sources and does not claim unseen-style transfer. The separate equality operation is withheld from training, with overlapping operands explicitly allowed.


![Reference plot: Reference solution 1. Axis labels identify the mechanism quantities.](../../figures/chapter-09/cs336-07_posttraining_integration-01.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 2: execute the actual ancestry

### Prediction

Before running, predict which checkpoint must initialize every sibling. Is matching a seed enough if its vocabulary or response mask differs? Run the entire frozen experiment, check every parent hash, and compare the first sampled pool across siblings. Preserve all seeds, not the most favorable one.


### Reference solution 2

Run the bounded reference below, then compare it with your prediction.


In [ ]:
result = pa.run_assignment()
print('Run:',result['status'],result['runtime_seconds'],'seconds; seeds:',[s['seed'] for s in result['seeds']])
for seed_result in result['seeds']:
    sft_hash=seed_result['ancestry']['coverage-sft']['final_state_sha256']
    print('Seed:',seed_result['seed'],'interface:',seed_result['interface_sha256'],'frozen:',seed_result['frozen_reference_checks'])
    for arm in protocol['sibling_arms']:
        assert seed_result['ancestry'][arm]['initial_state_sha256']==sft_hash
        print(arm,seed_result['ancestry'][arm])
    pools=[seed_result['stages'][arm]['rollouts'][0]['response_ids'] for arm in protocol['sibling_arms']]
    assert pools[0]==pools[1]==pools[2]
    print('Same first pool; later fresh collections may diverge. Signal:',seed_result['signal_gate'])


### Reference explanation 2

The random parent initializes both SFT arms with identical actual weights. Every sibling starts from the coverage-SFT state, while that state is copied into a frozen reference. Hash checks protect both frozen parents. The first sibling pool shares the parent and RNG seed; later pools are freshly sampled from the diverging policies. Equal update caps do not make all later responses or accepted label exposures equal.

```mermaid
flowchart LR
  P[Random compatible CPU parent] --> C[Coverage SFT: 8 sources]
  P --> R[Repetition SFT: 2 sources]
  C --> A[Accepted-response SFT]
  C --> G[GRPO population std]
  C --> M[GRPO center-only]
```

This diagram is the assignment's actual lineage. It is distinct from the book's retained TinyStories, Qwen Base/SFT/preference, and acquired Instruct/RLVR roots. The CLI stores complete checkpoints with interface/protocol hashes; `load_checkpoint` rejects a foreign interface. The notebook executes the same bounded consumer in memory without overwriting the retained report.


## Checkpoint 3: coverage versus repetition and paired templates

### Prediction

Predict whether broader coverage must win for every seed and every delivery metric. Compare paired held-out sources in both template forms, then contrast answer accuracy, termination, and complete-path probability with the always-terminated constant baseline. Does a better complete path establish parity learning?


### Reference solution 3

Run the bounded reference below, then compare it with your prediction.


In [ ]:
stages=['coverage-sft','repetition-sft','accepted-sft','grpo-std','grpo-center']
fig,axes=plt.subplots(1,2,figsize=(13,4))
for sr in result['seeds']:
    values=[]
    for stage in stages:
        ev=sr['stages'][stage]['evaluation']
        ids=[i['id'] for i in items if i['split']=='heldout-source']
        values.append(sum(ev['exact_complete_path_probability'][key] for key in ids)/len(ids))
    axes[0].plot(range(len(stages)),values,marker='o',label=f"seed{sr['seed']}")
    print('All exact heldout probabilities:',sr['seed'],dict(zip(stages,values)))
    print('Coverage−repetition paired source deltas:',sr['coverage_vs_repetition']['summaries']['heldout-source'])
axes[0].axhline(.5,color='black',ls='--',label='Terminated constant baseline')
axes[0].set_xticks(range(len(stages)),stages,rotation=20)
axes[0].set(ylabel='Exact conditional complete-path probability',title='Same paired held-out sources, all seeds',ylim=(0,1))
axes[0].legend(fontsize=8)
for metric in ['answer_correct','natural_termination','complete_path_success']:
    values=[]
    for stage in stages:
        rows=[r for sr in result['seeds'] for r in sr['stages'][stage]['evaluation']['rows'] if r['split']=='heldout-source' and r['decoding']=='sampled']
        values.append(sum(r[metric] for r in rows)/len(rows))
    axes[1].plot(range(len(stages)),values,marker='o',label=metric)
axes[1].set_xticks(range(len(stages)),stages,rotation=20)
axes[1].set(ylabel='Fixed-seeded sampled fraction',title='Separate answer, stop, and delivered response',ylim=(0,1)); axes[1].legend(fontsize=8)
plt.tight_layout();plt.show()
for sr in result['seeds']:
    for stage in ['coverage-sft','repetition-sft']:
        print(sr['seed'],stage,{k:v for k,v in sr['stages'][stage]['evaluation']['summaries'].items() if k.startswith('heldout-source/')})


### Reference explanation 3

The left plot is an exact two-step conditional path probability, calculated from each checkpoint; the right plot is the actual fixed-seeded sampled response fraction. They are different instruments. Both preserve all seeds and every held-out source/template row. Source-level paired deltas report regressions and descriptive standard errors without treating the two templates as independent problems.

The retained reference run has broader coverage exceeding repetition in held-out complete probability for all3 seeds, with a much smaller difference in seed33672. Reward siblings raise complete probability toward.5 while answer-only performance remains weak; the always-terminated constant policy still provides an essential.5 baseline. Improving EOS can improve delivered responses without establishing parity generalization. Inspect the actual freshly computed rows before making the claim; no best checkpoint selection occurs.


![Reference plot: Reference solution 3. Axis labels identify the mechanism quantities.](../../figures/chapter-09/cs336-07_posttraining_integration-02.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 4: inspect all attempted groups and gradient components

### Prediction

Can nonzero total gradient prove that the reward task taught anything? Predict the relation between all-right/all-wrong groups, mixed fraction, task-advantage norm, task-gradient norm, and KL-gradient norm. Inspect the first eight actual sampled responses. Also account for discarded work if someone were to retain only mixed groups.


### Reference solution 4

Run the bounded reference below, then compare it with your prediction.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for sr,ax in zip(result['seeds'],axes):
    history=sr['stages']['grpo-std']['history']
    ax.plot([h['update'] for h in history],[h['task_reward_gradient_norm'] for h in history],label='Task reward')
    ax.plot([h['update'] for h in history],[h['weighted_kl_gradient_norm'] for h in history],label='Weighted KL')
    ax.set(xlabel='Fresh-rollout optimizer update',ylabel='Gradient L2 norm',title=f"Seed{sr['seed']}");ax.set_yscale('symlog',linthresh=1e-8);ax.legend(fontsize=8)
    print('Eligibility:',sr['seed'],sr['difficulty_selection']['eligible_prompt_ids'],'discarded:',sr['difficulty_selection']['discarded_prompt_ids'])
    print('Train/dev signal:',sr['signal_gate'])
    first=sr['stages']['grpo-std']['rollouts'][0]
    for j in range(8):
        print(first['item_ids'][j],first['response_ids'][j],first['grades'][j],first['advantages'][j],first['old_conditional_logp'][j])
    print('Counterfactual mixed-filter population/cost:',history[0]['mixed_filter_counterfactual'])
    print('First/last gradient components:',{k:history[0][k] for k in ('task_reward_gradient_norm','weighted_kl_gradient_norm','task_kl_gradient_dot','mixed_group_fraction')},{k:history[-1][k] for k in ('task_reward_gradient_norm','weighted_kl_gradient_norm','task_kl_gradient_dot','mixed_group_fraction')})
plt.tight_layout();plt.show()


### Reference explanation 4

The task and weighted-KL components are measured independently with autograd before clipping. Their vector dot product and combined norm are retained; their norms alone cannot describe reinforcement versus opposition. A fresh behavior/current score match is checked at every update, using the exact same grammar support, path IDs, prompt, and evaluation mode. Old scores, advantages, and the reference remain detached.

The fixed training-only difficulty policy retains prompts with parent complete-path probability between.05 and.95. The current reference retained all16 training template rows; this is an actual outcome, not a fabricated selection shift. Its train/dev collections contain both successes and failures. The separate mixed-filter counterfactual reports discarded groups and the source counts among retained groups; all attempts remain in the real objective. The reward-audit notebook explains why conditioning on mixed groups would change both population and estimator.


![Reference plot: Reference solution 4. Axis labels identify the mechanism quantities.](../../figures/chapter-09/cs336-07_posttraining_integration-03.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 5: equal attempts are unequal objectives

### Prediction

The siblings receive the same first sampled pool and16-update caps. Predict which budgets match and which do not. Does accepted-response SFT learn from failed responses? Contrast its actual CE gradient with the reward arm's signed task advantages, then inspect the later pools without assuming they remain paired.


### Reference solution 5

Run the bounded reference below, then compare it with your prediction.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for sr in result['seeds']:
    names=protocol['sibling_arms']
    attempts=[sum(h['attempted_responses'] for h in sr['stages'][a]['history']) for a in names]
    labels=[sr['stages'][a]['accepted_supervised_tokens'] for a in names]
    axes[0].plot(range(3),attempts,marker='o',label=f"seed{sr['seed']}")
    axes[1].plot(range(3),labels,marker='o',label=f"seed{sr['seed']}")
    for arm in names:
        stage=sr['stages'][arm]
        print(sr['seed'],arm,'updates',len(stage['history']),'response tokens',stage['attempted_response_tokens'],'accepted SFT labels',stage['accepted_supervised_tokens'])
        print('Clock example:',{k:stage['history'][0][k] for k in ('physical_generation_positions','physical_generation_prefix_tokens','physical_objective_scoring_input_tokens','accepted_sft_forward_input_tokens','actual_objective','actual_objective_gradient_norm_before_clip')})
for ax in axes: ax.set_xticks(range(3),protocol['sibling_arms'],rotation=15);ax.legend(fontsize=8)
axes[0].set(ylabel='Attempted sampled responses',title='Predeclared attempts and update caps match')
axes[1].set(ylabel='Accepted supervised response labels',title='Accepted-SFT exposure is outcome-dependent')
plt.tight_layout();plt.show()


### Reference explanation 5

Each main sibling attempts16×16×8=2048 responses and4096 generated response positions in the current run. Accepted-response SFT uses only actual complete correct sampled paths, with two supervised labels each. It never replaces a failed path by an oracle answer. Its actual objective is accepted-token-mean full-vocabulary CE, with no KL term; the recorded GRPO task/KL gradient is a labeled counterfactual diagnostic.

Reward arms include every eligible response, including signed advantages for relative failures. They use conditional response-mean clipping and exact categorical KL on sampled prefix states. Center-only and population-std normalization reuse the same kernel; they differ in scaling and finite-group statistics. Matched attempts do not match accepted labels, physical scoring, or all future rollouts. The figure exposes those differences instead of calling the arms equal compute.


![Reference plot: Reference solution 5. Axis labels identify the mechanism quantities.](../../figures/chapter-09/cs336-07_posttraining_integration-04.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 6: change the stop and sampling support

### Prediction

Hold a coverage-SFT checkpoint fixed. Predict what a one-token cap does to answer-only accuracy and completion, and what removing the grammar does to the delivered response. Would appending EOS after collection prove learned termination? Display a labeled manufactured-EOS counterfactual without changing the real rows or rewards.


### Reference solution 6

Run the bounded reference below, then compare it with your prediction.


In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
conditions=['conditional cap2','conditional cap1','full-vocabulary cap2']
for metric in ['answer_correct','natural_termination','complete_path_success']:
    values=[]
    for condition in conditions:
        rows=[]
        for sr in result['seeds']:
            stage=sr['stages']['coverage-sft']
            ev=stage['evaluation'] if condition=='conditional cap2' else stage['cap1'] if condition=='conditional cap1' else stage['full_vocabulary']
            rows += [r for r in ev['rows'] if r['split']=='heldout-source' and r['decoding']=='greedy']
        values.append(sum(r[metric] for r in rows)/len(rows))
    ax.plot(range(3),values,marker='o',label=metric)
ax.set_xticks(range(3),conditions);ax.set(ylabel='Greedy fraction across all seeds and held-out template pairs',ylim=(0,1),title='Fixed checkpoint: stopping and support interventions')
ax.legend(fontsize=8);plt.tight_layout();plt.show()
short=[r for r in result['seeds'][0]['stages']['coverage-sft']['cap1']['rows'] if r['split']=='heldout-source']
print('Actual cap1 paths:',[(r['item_id'],r['token_ids'],r['stop_reason']) for r in short])
print('Manufactured-EOS illustration (not generated, not rewarded):',[(r['item_id'],r['token_ids']+[2]) for r in short])
assert all(not r['natural_termination'] and not r['complete_path_success'] for r in short)


### Reference explanation 6

Cap1 prevents EOS from being sampled because the grammar's first support contains only answers. It can preserve an answer while eliminating every naturally terminated complete response. Appending EOS would manufacture a different path and cannot enter the reward record. The printed counterfactual is deliberately labeled.

Full-vocabulary generation removes the conditional syntax restriction. Its scores and trajectories answer a different question; neither its outcomes nor grammar-conditional improvements imply unrestricted English capability. SFT uses full-vocabulary CE throughout, whereas the RL likelihood, entropy, and KL refer to the stated conditional support. The integration contract makes that difference explicit instead of passing weights between incompatible microscopes.


![Reference plot: Reference solution 6. Axis labels identify the mechanism quantities.](../../figures/chapter-09/cs336-07_posttraining_integration-05.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 7: defend transfer and negative coverage

### Prediction

Write a claim supported by the experiment and a claim it cannot support. Predict whether sibling gains must carry to the independently withheld equality operation. Use source-paired regressions, all seeds, and the original verifier replay; identify a negative case rather than selecting only a successful response.


### Reference solution 7

Run the bounded reference below, then compare it with your prediction.


In [ ]:
for sr in result['seeds']:
    print('Seed',sr['seed'])
    for arm in protocol['sibling_arms']:
        paired=sr['stages'][arm]['paired_vs_parent']
        print(arm,{split:summary for split,summary in paired['summaries'].items() if split in ('heldout-source','heldout-family')})
    rows=[r for r in sr['stages']['grpo-std']['evaluation']['rows'] if r['split']=='heldout-family' and not r['complete_path_success']]
    print('First retained unseen-family failure:',rows[0] if rows else 'No failure in this bounded sampled collection; still retain every row.')
print('Historical strict replay:',result['native_reward_audit']['summary'])
print('Frozen protocol:',result['protocol_sha256'],'all source hashes:',result['source_sha256'])
print('Evidence boundary: symbolic decoder; conditional grammar; tiny paired source panel; no native launch, no English capability claim.')


### Reference explanation 7

A defensible claim distinguishes mechanism signal, delivery improvement, and task transfer. The new balanced symbolic task provides genuine successes/failures and task-gradient evidence that the original all-zero native reward pilots lacked. It also preserves weak transfer and the constant-baseline comparison. A falling loss, a growing complete-path probability, or a nonzero total gradient alone is insufficient for a capability claim.

The equality panel is an independently withheld task operation, not merely a spelling variation. Its negatives remain in the summaries and raw rows. The paired-source errors and standard errors are descriptive for the frozen small panel; template rows are clustered by source. No held-out result changes the protocol, caps, or selected checkpoint.

The actual retained command, all seed checkpoints, raw sampled paths, gradient components, archive source hashes, and frozen precollection spec live under `experiments/reports/2026-10-11-cs336-upgrade/run-01/posttraining/`. Read the [worked solution and acceptance boundaries](../../../book/solutions/cs336-posttraining-assignment.md). Native SFT/RL follow-through is a separately gated protocol; this notebook launches none.
